# BARAM 2026 — 신경망 bake-off (ResNet-MLP · 1D-CNN vs 현행 MLP)

비용 무제한 모드. 현행 MLP(블렌드 70%)의 **대체/보강** 후보를 2폴드 규율로 판정.

- **ResNet-MLP**: RTDL(Gorishniy et al.) 스타일 — skip connection + BatchNorm. 정형 벤치마크에서 plain MLP 반발짝 우위.
- **1D-CNN**: feature 벡터 위 Conv1d — 유도편향 다양성.
- 평가 방식 2가지: (a) **대체** — MLP 자리에 그대로, (b) **혼합** — MLP와 50:50 후 GBM과 블렌드.
- 설정: `ficrw_result.json`의 채택 α·MLP cfg를 런타임에 로드(없으면 v7 기본값).
- 판정: 두 폴드 모두 현행 이상. 시드 3(판정용), 채택 시 최종은 시드 10.

## 0. 설정

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS","1")
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd
import torch, torch.nn as nn
torch.set_num_threads(1)
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
import wind_lib as W
from official_metric import group_scores
from pathlib import Path

DEV="mps" if torch.backends.mps.is_available() else "cpu"
SEEDS=[15,0,1]; BLEND=0.7; STUCK_TH=0.3; STUCK_W=0.5
# 채택 결과 로드 (없으면 v7 기본)
ALPHA=0.0; MLP_CFG=dict(h=256,depth=3,drop=0.0,lr=0.0015868563457489381,wd=1e-4,bs=256,emb=4)
if Path("submission/ver_8/final_alpha.json").exists():
    ALPHA=float(json.load(open("submission/ver_8/final_alpha.json"))["final_alpha"])
elif Path("submission/ver_8/ficrw_result.json").exists():
    ALPHA=float(json.load(open("submission/ver_8/ficrw_result.json")).get("adopted_alpha",0.0))
print(f"α={ALPHA}  MLP_CFG={MLP_CFG}")
GBM_PARAMS=dict(objective="mae",n_estimators=2000,learning_rate=0.020651822836313095,
    num_leaves=63,min_child_samples=300,subsample=0.8,subsample_freq=1,colsample_bytree=0.5,
    reg_lambda=0.1,random_state=42,n_jobs=1,verbose=-1)
RAW=os.environ.get("WIND_RAW_DIR",os.path.expanduser("~/Downloads/open"))
GROUPS=(1,2,3)
FR,TGT={},{}
for g in GROUPS:
    df,tgt=W.load_train(g); TGT[g]=tgt
    FR[g]=W.add_spatial(W.build(df,g),"train")
BASE_ALL=[c for c in W.feature_cols(FR[1]) if c not in W.SPATIAL_COLS]+["pc_pred_cf"]
FEATS=W.lean_features(BASE_ALL)+W.SPATIAL_COLS

def stuck_frac():
    frames={}
    for fn,pre,n,rate in [("scada_vestas_train.csv","vestas",12,3600.0),
                          ("scada_unison_train.csv","unison",5,4200.0)]:
        d=pd.read_csv(f"{RAW}/train/{fn}",encoding="utf-8-sig",parse_dates=["kst_dtm"])
        d["hour"]=d["kst_dtm"].dt.ceil("h")
        agg={}
        for i in range(1,n+1):
            pw=f"{pre}_wtg{i:02d}_power_kw10m"; ws=f"{pre}_wtg{i:02d}_ws"
            h=d.groupby("hour").agg(pw_m=(pw,"mean"),ws_m=(ws,"mean"))
            agg[i]=pd.DataFrame({f"stuck_{i}":((h.ws_m>=4.0)&(h.pw_m<=0.01*rate)).astype(float),
                                 f"rep_{i}":h.pw_m.notna().astype(float)})
        frames[pre]=pd.concat(agg.values(),axis=1)
    def frac(pre,ids):
        f=frames[pre]
        st=f[[f"stuck_{i}" for i in ids]].sum(axis=1); rp=f[[f"rep_{i}" for i in ids]].sum(axis=1)
        return (st/rp).where(rp>=3)
    return {1:frac("vestas",range(1,7)),2:frac("vestas",range(7,13)),3:frac("unison",range(1,6))}
FRAC=stuck_frac()
for g in GROUPS:
    s=FRAC[g].reindex(FR[g].kst_dtm).to_numpy()
    FR[g]=FR[g].assign(stuck_frac=np.nan_to_num(s,nan=0.0))
def make_weight(fr,tgt,cap):
    w=np.where(fr["stuck_frac"]>=STUCK_TH,STUCK_W,1.0)
    if ALPHA>0: w=w*(1.0+ALPHA*np.clip(fr[tgt].to_numpy()/cap,0,1))
    return w

FOLDS={2023:[2022],2024:[2022,2023]}
CACHE={}
for vy,tys in FOLDS.items():
    ent={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]; yr=fr.kst_dtm.dt.year
        tr=fr[yr.isin(tys)]; va=fr[yr==vy]
        if len(tr)==0 or len(va)==0: continue
        iso=W.fit_powercurve(tr,tgt,cap)
        ent[g]=(W.with_pc(tr,iso),W.with_pc(va,iso))
    CACHE[vy]=ent

# GBM 폴드 예측 (1회 계산)
GBM={}
for vy,ent in CACHE.items():
    d={}
    for g,(tr2,va2) in ent.items():
        cap=W.CAP[g]; tgt=TGT[g]; w=make_weight(tr2,tgt,cap)
        lg_=lgb.LGBMRegressor(**GBM_PARAMS).fit(tr2[FEATS],tr2[tgt],sample_weight=w)
        hg_=HGB(loss="absolute_error",max_iter=600,learning_rate=0.03,max_leaf_nodes=63,
            l2_regularization=1.0,random_state=42).fit(tr2[FEATS].to_numpy(),tr2[tgt].to_numpy(),sample_weight=w)
        d[g]=np.clip(0.5*(lg_.predict(va2[FEATS])+hg_.predict(va2[FEATS].to_numpy())),0,cap)
    GBM[vy]=d
print("GBM cached")

α=5.0  MLP_CFG={'h': 256, 'depth': 3, 'drop': 0.0, 'lr': 0.0015868563457489381, 'wd': 0.0001, 'bs': 256, 'emb': 4}


GBM cached


## 1. 신경망 3종 정의 (공통 학습 루프)

In [2]:
class MLP(nn.Module):
    def __init__(s,nf,ng=3,h=256,depth=3,drop=0.0,emb=4,**kw):
        super().__init__(); s.emb=nn.Embedding(ng,emb)
        L=[nn.Linear(nf+emb,h),nn.GELU(),nn.Dropout(drop)]
        for _ in range(depth-1): L+=[nn.Linear(h,h),nn.GELU(),nn.Dropout(drop)]
        L+=[nn.Linear(h,1)]; s.net=nn.Sequential(*L)
    def forward(s,x,g): return s.net(torch.cat([x,s.emb(g)],-1)).squeeze(-1)

class ResBlock(nn.Module):
    def __init__(s,h,drop):
        super().__init__()
        s.bn=nn.BatchNorm1d(h); s.l1=nn.Linear(h,h*2); s.l2=nn.Linear(h*2,h)
        s.act=nn.GELU(); s.drop=nn.Dropout(drop)
    def forward(s,x): return x+s.l2(s.drop(s.act(s.l1(s.bn(x)))))

class ResNetMLP(nn.Module):
    """RTDL 스타일: 입력 프로젝션 + residual block × depth."""
    def __init__(s,nf,ng=3,h=256,depth=3,drop=0.1,emb=4,**kw):
        super().__init__(); s.emb=nn.Embedding(ng,emb)
        s.inp=nn.Linear(nf+emb,h)
        s.blocks=nn.ModuleList([ResBlock(h,drop) for _ in range(depth)])
        s.head=nn.Sequential(nn.BatchNorm1d(h),nn.GELU(),nn.Linear(h,1))
    def forward(s,x,g):
        z=s.inp(torch.cat([x,s.emb(g)],-1))
        for b in s.blocks: z=b(z)
        return s.head(z).squeeze(-1)

class CNN1D(nn.Module):
    """feature 벡터를 1D 시퀀스로 보고 Conv — 유도편향 다양성."""
    def __init__(s,nf,ng=3,ch=64,emb=4,drop=0.1,**kw):
        super().__init__(); s.emb=nn.Embedding(ng,emb); s.nf=nf+emb
        s.net=nn.Sequential(
            nn.Conv1d(1,ch,5,padding=2), nn.GELU(), nn.Dropout(drop),
            nn.Conv1d(ch,ch,5,padding=2), nn.GELU())
        s.head=nn.Sequential(nn.Linear(ch,128), nn.GELU(), nn.Dropout(drop), nn.Linear(128,1))
    def forward(s,x,g):
        z=torch.cat([x,s.emb(g)],-1).unsqueeze(1)
        return s.head(s.net(z).mean(-1)).squeeze(-1)   # mean-pool (MPS AdaptiveAvgPool 비호환 회피)

ARCH={"mlp":MLP,"resnet":ResNetMLP,"cnn":CNN1D}

def train_nn(pool_tr,seed,arch):
    torch.manual_seed(seed); np.random.seed(seed)
    pool_tr=pool_tr.sort_values("kst_dtm")
    mu=pool_tr[FEATS].mean(); sd=pool_tr[FEATS].std()+1e-8
    X=((pool_tr[FEATS]-mu)/sd).to_numpy(np.float32)
    y=pool_tr["cf"].to_numpy(np.float32); gid=pool_tr["gid"].to_numpy(np.int64)
    wt=pool_tr["w"].to_numpy(np.float32)
    n=len(X); cut=int(n*0.85)
    Xt=torch.tensor(X,device=DEV); yt=torch.tensor(y,device=DEV)
    gt=torch.tensor(gid,device=DEV); wtt=torch.tensor(wt,device=DEV)
    kw=dict(MLP_CFG) if arch=="mlp" else dict(h=256,depth=3,drop=0.1,emb=4,lr=1e-3,wd=1e-4,bs=256)
    lr=kw.pop("lr",1e-3); wd=kw.pop("wd",1e-4); bs=int(kw.pop("bs",256))
    net=ARCH[arch](len(FEATS),**kw).to(DEV)
    opt=torch.optim.AdamW(net.parameters(),lr=lr,weight_decay=wd)
    best=1e9; st=None; pat=0
    for ep in range(120):
        net.train(); perm=np.random.permutation(np.arange(cut))
        for i in range(0,len(perm),bs):
            b=torch.tensor(perm[i:i+bs],device=DEV); opt.zero_grad()
            e=(net(Xt[b],gt[b])-yt[b]).abs()
            ((e*wtt[b]).sum()/(wtt[b].sum()+1e-8)).backward(); opt.step()
        net.eval()
        with torch.no_grad():
            e=(net(Xt[cut:],gt[cut:])-yt[cut:]).abs()
            vl=((e*wtt[cut:]).sum()/(wtt[cut:].sum()+1e-8)).item()
        if vl<best-1e-5: best=vl; st={k:v.clone() for k,v in net.state_dict().items()}; pat=0
        else: pat+=1
        if pat>=10: break
    net.load_state_dict(st); return net,(mu,sd)

def predict_nn(net,scaler,fr,g,cap):
    mu,sd=scaler
    X=torch.tensor(((fr[FEATS]-mu)/sd).to_numpy(np.float32),device=DEV)
    gid=torch.full((len(fr),),g-1,dtype=torch.long,device=DEV)
    net.eval()
    with torch.no_grad(): p=net(X,gid).cpu().numpy()
    return np.clip(p,0,1)*cap

def nn_fold_preds(arch):
    out={}
    for vy,ent in CACHE.items():
        pool=[]
        for g,(tr2,_) in ent.items():
            p=tr2[FEATS+["kst_dtm"]].copy()
            p["cf"]=tr2[TGT[g]]/W.CAP[g]; p["gid"]=g-1
            p["w"]=make_weight(tr2,TGT[g],W.CAP[g]); pool.append(p)
        pool=pd.concat(pool,ignore_index=True)
        acc={g:[] for g in ent}
        for sd_ in SEEDS:
            net,scaler=train_nn(pool,sd_,arch)
            for g,(_,va2) in ent.items():
                acc[g].append(predict_nn(net,scaler,va2,g,W.CAP[g]))
        out[vy]={g:np.mean(v,axis=0) for g,v in acc.items()}
    return out

def total(vy,nn_preds,B=BLEND):
    nm=[]; fi=[]
    for g,(_,va2) in CACHE[vy].items():
        cap=W.CAP[g]
        p=np.clip((1-B)*GBM[vy][g]+B*nn_preds[vy][g],0,cap)
        a,b=group_scores(va2[TGT[g]].to_numpy(),p,cap); nm.append(a); fi.append(b)
    return 0.5*(1-np.mean(nm))+0.5*np.mean(fi)

## 2. bake-off 실행 & 판정

In [3]:
PREDS={}
for arch in ["mlp","resnet","cnn"]:
    PREDS[arch]=nn_fold_preds(arch)
    print(f"{arch:8s}: 2023={total(2023,PREDS[arch]):.4f}  2024={total(2024,PREDS[arch]):.4f}")

# 혼합 변형: mlp와 50:50
for other in ["resnet","cnn"]:
    mix={vy:{g:0.5*(PREDS["mlp"][vy][g]+PREDS[other][vy][g]) for g in PREDS["mlp"][vy]} for vy in CACHE}
    PREDS[f"mlp+{other}"]=mix
    print(f"mlp+{other:6s}: 2023={total(2023,mix):.4f}  2024={total(2024,mix):.4f}")
# 3종 전부
mix3={vy:{g:(PREDS["mlp"][vy][g]+PREDS["resnet"][vy][g]+PREDS["cnn"][vy][g])/3 for g in PREDS["mlp"][vy]} for vy in CACHE}
PREDS["mlp+resnet+cnn"]=mix3
print(f"3종 혼합  : 2023={total(2023,mix3):.4f}  2024={total(2024,mix3):.4f}")

cur23,cur24=total(2023,PREDS["mlp"]),total(2024,PREDS["mlp"])
cands=[]
for k,pp in PREDS.items():
    if k=="mlp": continue
    t23,t24=total(2023,pp),total(2024,pp)
    if t23>=cur23 and t24>=cur24: cands.append((k,min(t23,t24),t23,t24))
if cands:
    k,mn,t23,t24=max(cands,key=lambda x:x[1])
    print(f"\n채택: {k} → 2023={t23:.4f} 2024={t24:.4f} (현행 mlp {cur23:.4f}/{cur24:.4f})")
    winner=k
else:
    print(f"\n현행 mlp 유지 (어느 변형도 두 해 모두 못 이김)")
    winner="mlp"
summary=dict(totals={k:{"2023":round(total(2023,pp),4),"2024":round(total(2024,pp),4)} for k,pp in PREDS.items()},
             winner=winner, alpha=ALPHA)
json.dump(summary,open("submission/ver_8/bakeoff_result.json","w"),ensure_ascii=False,indent=2)
print(json.dumps(summary,ensure_ascii=False,indent=2))

mlp     : 2023=0.6193  2024=0.6327


resnet  : 2023=0.6048  2024=0.6299


cnn     : 2023=0.5976  2024=0.6306
mlp+resnet: 2023=0.6144  2024=0.6320
mlp+cnn   : 2023=0.6134  2024=0.6351
3종 혼합  : 2023=0.6104  2024=0.6337

현행 mlp 유지 (어느 변형도 두 해 모두 못 이김)
{
  "totals": {
    "mlp": {
      "2023": 0.6193,
      "2024": 0.6327
    },
    "resnet": {
      "2023": 0.6048,
      "2024": 0.6299
    },
    "cnn": {
      "2023": 0.5976,
      "2024": 0.6306
    },
    "mlp+resnet": {
      "2023": 0.6144,
      "2024": 0.632
    },
    "mlp+cnn": {
      "2023": 0.6134,
      "2024": 0.6351
    },
    "mlp+resnet+cnn": {
      "2023": 0.6104,
      "2024": 0.6337
    }
  },
  "winner": "mlp",
  "alpha": 5.0
}
